# Import

In [2]:
import pandas as pd

from meridian.data import data_frame_input_data_builder as data_builder

from meridian.analysis.visualizer import MediaSummary
from meridian import constants

from meridian.model import model
from meridian.model import prior_distribution
from meridian.model import spec

import pandas as pd
# check if GPU is available
from psutil import virtual_memory
import tensorflow as tf
import tensorflow_probability as tfp




# Input data

### Artificial data

In [3]:
data = {
    'geo': ['GEO_01', 'GEO_01', 'GEO_01', 'GEO_01', 'GEO_01', 'GEO_01', 'GEO_01', 'GEO_01', 'GEO_01', 'GEO_01',
            'GEO_01', 'GEO_01', 'GEO_01', 'GEO_01', 'GEO_01', 'GEO_01', 'GEO_01', 'GEO_01', 'GEO_01', 'GEO_01',
            'GEO_02', 'GEO_02', 'GEO_02', 'GEO_02', 'GEO_02', 'GEO_02', 'GEO_02', 'GEO_02', 'GEO_02', 'GEO_02',
            'GEO_02', 'GEO_02', 'GEO_02', 'GEO_02', 'GEO_02', 'GEO_02', 'GEO_02', 'GEO_02', 'GEO_02', 'GEO_02',
            'GEO_03', 'GEO_03', 'GEO_03', 'GEO_03', 'GEO_03', 'GEO_03', 'GEO_03', 'GEO_03', 'GEO_03', 'GEO_03',
            'GEO_03', 'GEO_03', 'GEO_03', 'GEO_03', 'GEO_03', 'GEO_03', 'GEO_03', 'GEO_03', 'GEO_03', 'GEO_03'],
    'week': ['2025-01-06', '2025-01-13', '2025-01-20', '2025-01-27', '2025-02-03', '2025-02-10', '2025-02-17', '2025-02-24', '2025-03-03', '2025-03-10',
             '2025-03-17', '2025-03-24', '2025-03-31', '2025-04-07', '2025-04-14', '2025-04-21', '2025-04-28', '2025-05-05', '2025-05-12', '2025-05-19'] * 3,
    'conversions': [116, 127, 117, 164, 144, 128, 135, 144, 152, 129, 152, 136, 137, 142, 138, 133, 131, 142, 124, 141,
                    128, 121, 142, 140, 142, 142, 142, 148, 145, 141, 139, 153, 146, 138, 134, 137, 143, 139, 133, 152,
                    134, 138, 141, 132, 142, 143, 146, 133, 148, 151, 144, 136, 152, 145, 140, 143, 146, 130, 132, 146],
    'tv_spend': [5154, 5538, 6482, 14274, 7569, 12515, 11590, 9331, 11967, 8666, 7600, 11930, 12294, 13662, 9118, 13538, 13013, 12123, 11180, 9790,
                 8402, 13404, 10133, 14391, 9747, 12035, 11638, 10256, 9353, 14578, 14627, 13389, 6128, 14602, 10313, 14901, 14430, 14255, 7342, 6983,
                 11379, 9012, 9103, 10964, 6251, 9072, 10368, 13471, 6283, 11261, 13789, 10712, 12411, 7390, 6164, 9115, 10878, 10523, 9892, 7251],
    'tv_impressions': [148854, 166427, 187508, 432943, 239698, 389709, 357870, 272267, 379004, 268127, 232979, 358778, 388339, 414378, 274136, 399899, 396044, 356224, 320246, 298729,
                       254654, 400616, 302419, 433702, 293046, 363706, 349208, 296266, 271282, 437222, 444631, 408528, 179281, 446095, 329631, 448179, 425897, 423634, 210093, 213745,
                       353303, 287888, 284248, 331426, 186402, 266473, 321211, 412738, 192911, 344296, 407618, 321102, 364432, 214057, 181829, 264683, 331433, 308249, 314197, 203267],
    'radio_spend': [3730, 4448, 3520, 1030, 4675, 3276, 4761, 4423, 2973, 2674, 4332, 3596, 1254, 3532, 2481, 2383, 3688, 1401, 4098, 2885,
                    1110, 2244, 2703, 3425, 4922, 2043, 3179, 3097, 1996, 4303, 3112, 2752, 2112, 1297, 3883, 1261, 4355, 2637, 4514, 2307,
                    1888, 3426, 2526, 1573, 1761, 2480, 3957, 2291, 3594, 3970, 3827, 4120, 1693, 1912, 3003, 1735, 3907, 1650, 2495, 1544],
    'radio_impressions': [174727, 211056, 186529, 48248, 250799, 178105, 253425, 235255, 143361, 132347, 236096, 198074, 72783, 192356, 138344, 140973, 215208, 80829, 243449, 175653,
                          53248, 109448, 131318, 172217, 242469, 100612, 168324, 161310, 103419, 228420, 155456, 145891, 116050, 67862, 203048, 65195, 224435, 135781, 225073, 121180,
                          91148, 166749, 130158, 80597, 92742, 136254, 219472, 117591, 188016, 222894, 216798, 228194, 96297, 107740, 170120, 92988, 208248, 86703, 127176, 85326],
    'internet_spend': [5558, 7340, 4632, 4987, 3018, 4895, 6743, 7174, 7062, 7223, 3742, 7154, 7216, 7139, 5024, 7730, 6205, 7763, 5263, 6372,
                       2882, 4350, 7060, 6821, 7056, 6187, 3935, 6570, 5349, 7023, 6844, 3815, 2105, 7651, 3762, 5184, 7404, 5311, 3458, 3127,
                       3264, 5219, 7403, 3431, 6855, 6276, 7425, 4205, 2722, 3716, 6296, 4745, 6961, 4160, 3736, 6166, 5233, 5870, 4937, 2246],
    'internet_impressions': [537786, 757799, 464041, 474507, 293293, 495509, 681881, 741755, 714527, 730421, 369070, 716792, 732209, 707169, 506597, 798136, 637445, 802687, 529509, 655982,
                             266136, 398144, 687076, 658249, 684327, 587932, 372567, 628424, 519075, 696593, 672254, 366607, 200345, 747808, 358623, 520270, 720159, 543181, 356451, 307399,
                             315773, 521518, 721419, 342586, 673568, 639964, 757136, 425412, 277829, 377805, 637948, 495233, 719017, 408051, 371940, 612370, 526413, 577006, 502727, 227458]
}

df = pd.DataFrame(data)
df.head(5)

,geo,week,conversions,tv_spend,tv_impressions,radio_spend,radio_impressions,internet_spend,internet_impressions
0,GEO_01,2025-01-06,116,5154,148854,3730,174727,5558,537786
1,GEO_01,2025-01-13,127,5538,166427,4448,211056,7340,757799
2,GEO_01,2025-01-20,117,6482,187508,3520,186529,4632,464041
3,GEO_01,2025-01-27,164,14274,432943,1030,48248,4987,474507
4,GEO_01,2025-02-03,144,7569,239698,4675,250799,3018,293293


In [ ]:
df.

540

In [5]:
df = df.copy()
df["time"] = pd.to_datetime(df["week"])   # nový sloupec time

In [6]:
pop_map = {"GEO_01": 1_100_000,  # zvol libovolná realistická čísla
           "GEO_02": 1_250_000,
           "GEO_03":   950_000}
df["population"] = df["geo"].map(pop_map)

In [ ]:
""" # for one geo only
df = df[df['geo']=="GEO_01"]
df.head(5) """

' # for one geo only\ndf = df[df[\'geo\']=="GEO_01"]\ndf.head(5) '

In [8]:
df.head(5)

,geo,week,conversions,tv_spend,tv_impressions,radio_spend,radio_impressions,internet_spend,internet_impressions,time,population
0,GEO_01,2025-01-06,116,5154,148854,3730,174727,5558,537786,2025-01-06,1100000
1,GEO_01,2025-01-13,127,5538,166427,4448,211056,7340,757799,2025-01-13,1100000
2,GEO_01,2025-01-20,117,6482,187508,3520,186529,4632,464041,2025-01-20,1100000
3,GEO_01,2025-01-27,164,14274,432943,1030,48248,4987,474507,2025-01-27,1100000
4,GEO_01,2025-02-03,144,7569,239698,4675,250799,3018,293293,2025-02-03,1100000


# Create input for Meridian

In [ ]:
builder = (
    data_builder.DataFrameInputDataBuilder(kpi_type="non_revenue")
        .with_kpi(df, kpi_col="conversions")
        .with_population(df)                      
        .with_media(
            df,
            media_cols=["tv_impressions", "radio_impressions", "internet_impressions"],
            media_spend_cols=["tv_spend", "radio_spend", "internet_spend"],
            media_channels=["TV", "Radio", "Internet"]
        )
)

input_data = builder.build()  


c:\Users\Martin\.virtualenvs\venv_portfolio-hkOMkeu3\lib\site-packages\meridian\data\input_data.py:471: UserWarning: Consider setting custom priors, as kpi_type was specified as `non_revenue` with no `revenue_per_kpi` being set. Otherwise, the total media contribution prior will be used with `p_mean=0.4` and `p_sd=0.2`. Further documentation available at https://developers.google.com/meridian/docs/advanced-modeling/unknown-revenue-kpi-custom#set-total-paid-media-contribution-prior
  warnings.warn(


# Create model and PRIOR

In [10]:
roi_mu = 0.2  # Mu for ROI prior for each media channel.
roi_sigma = 0.9  # Sigma for ROI prior for each media channel.

prior = prior_distribution.PriorDistribution(
    roi_m=tfp.distributions.LogNormal(roi_mu, roi_sigma, name=constants.ROI_M)
)

model_spec = spec.ModelSpec(prior=prior)
mmm = model.Meridian(input_data=input_data, model_spec=model_spec)


c:\Users\Martin\.virtualenvs\venv_portfolio-hkOMkeu3\lib\site-packages\meridian\model\model.py:874: UserWarning: Consider setting custom ROI priors, as kpi_type was specified as `non_revenue` with no `revenue_per_kpi` being set. Otherwise, the total media contribution prior will be used with `p_mean=0.4` and `p_sd=0.2`. Further documentation available at  https://developers.google.com/meridian/docs/advanced-modeling/unknown-revenue-kpi-custom#set-total-paid-media-contribution-prior
  warnings.warn(


# Run MCMC (Markov Chain Monte Carlo)

In [11]:
mmm.sample_prior(200) 

mmm.sample_posterior(
    n_chains=2,
    n_adapt=800,
    n_burnin=200,
    n_keep=400,
    seed=42
)

# Get results

In [12]:


media_summary = MediaSummary(mmm)


In [13]:
df = media_summary.summary_table(
    include_prior=False,
    include_posterior=True,
    include_non_paid_channels=False
)

df

c:\Users\Martin\.virtualenvs\venv_portfolio-hkOMkeu3\lib\site-packages\meridian\analysis\visualizer.py:1629: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  .aggregate(lambda g: f'{g[0]} ({g[1]}, {g[2]})')


,channel,distribution,impressions,% impressions,spend,% spend,cpm,incremental KPI,% contribution,roi,effectiveness,mroi,cpik
0,TV,posterior,"19,105,756",31%,"633,530",56%,$33,"249,976 (124,095, 375,855)","18% (9%, 27%)","0.4 (0.2, 0.6)","0.01 (0.01, 0.02)","0.3 (0.2, 0.5)","$3.4 ($1.7, $5.1)"
1,Radio,posterior,"9,303,584",15%,"175,764",15%,$19,"147,008 (106,268, 187,752)","11% (8%, 14%)","0.8 (0.6, 1.1)","0.02 (0.01, 0.02)","0.7 (0.5, 0.9)","$1.3 ($0.9, $1.7)"
2,Internet,posterior,"32,769,902",54%,"329,000",29%,$10,"972,658 (528,736, 1,416,552)","71% (38%, 103%)","3.0 (1.6, 4.3)","0.03 (0.02, 0.04)","2.3 (1.0, 3.7)","$0.4 ($0.2, $0.6)"
3,All Channels,posterior,"61,179,240",100%,"1,138,294",100%,$19,"1,369,630 (759,099, 1,980,159)","99% (55%, 144%)","1.2 (0.7, 1.7)","nan (nan, nan)","nan (nan, nan)","$1.0 ($0.6, $1.5)"


In [ ]:
''' 
How to read results:

conversion =  action that can lead to profit

Return on Investment:
ROI = conversions/spend
roi = 0.4 (0.2, 0.6)
    0.4 = average estimated ROI (posterior mean)
    (0.2, 0.6) = 95% credible interval, i.e., ROI is likely between 0.2 and 0.6


mroi =  If you invest one more dollar, you'll get xxx additional conversions (with a 95% credible interval between x1 and x2 conversions)

CPiK =  Cost Per Incremental KPI = How much money did I have to spend to get one additional conversion

CPiK = 1/ROI

'''